# Catalyst Surface Adsorption Configuration Search with NVIDIA ALCHEMI

![Catalyst adsorption configuration search](assets/banner_adsorbml_bgr.svg)

*Interactive tutorial; small-panel mode is intended for smoke testing, while the full panel is sized for a single data-center GPU.*

## Chemical discovery starts as a search problem

Chemical discovery begins with an enormous solution space. A research goal can branch across composition, molecular structure, crystal structure, interfaces, process conditions, charge state, environment, target property, and practical constraints such as cost, safety, synthesis route, or lifetime. The first step is therefore not to run a model. It is to define a meaningful search space.

A useful computational workflow narrows that space in stages:

- **Frame the scientific question.** Decide what is being optimized or explained: stability, selectivity, conductivity, degradation, binding, diffusion, emission, or another property.
- **Apply chemical judgement.** Use known chemistry, physics, synthesis constraints, and domain intuition to remove candidates that are irrelevant or impossible for the intended application.
- **Define the candidate set.** Turn the remaining idea into structures that can be simulated: molecules, conformers, crystals, defects, interfaces, reaction states, adsorption geometries, or other local atomic arrangements.
- **Choose the evidence ladder.** Decide which results are screening signals, which require DFT or experiment, and what metadata is needed before two numbers can be compared.

Even after this narrowing, the candidate set can still contain hundreds, thousands, or millions of structures. DFT and experiment remain essential for final validation, but they are usually too expensive for the full exploratory loop.

Machine-learned interatomic potentials enter at this point as accelerators for atomistic simulation. Their role is not just to find a better placement for one adsorbate. More generally, they make it practical to relax, rank, and filter large sets of atomic structures so that the expensive validation budget is spent on better chosen cases. In that sense, the model is the last computational ingredient inside a broader discovery workflow: first define the question and candidate space, then choose the model and execution path that can evaluate it.

<mark>TODO - CODEX</mark>: Keep this opening general. Move AdsorbML-specific panel details, sites, coverages, and reference rows into the next scientific-context cells.

<mark>TODO - HUMAN</mark>: Confirm which discovery domains to mention explicitly beyond adsorption: batteries, OLEDs, polymers, catalysts, molecular stability, or none.

<mark>TODO - VISUAL AGENT</mark>: Render a general discovery-funnel graphic: broad chemical space -> domain narrowing -> candidate structures -> accelerated simulation -> ranked outputs -> DFT/experiment validation.

## Where ALCHEMI fits

ALCHEMI is the acceleration layer for the atomistic-simulation part of chemical and materials discovery. Its main value is that these workflows become **batched**, **customizable**, and **GPU-native**. Models such as MACE, AIMNet2, and TensorNet plug into that stack, but the ecosystem story is much broader than choosing a model.

For this tutorial, the relevant ALCHEMI layers are:

- <span style="color:#76b900"><strong>Toolkit / NIM</strong></span> **Batched execution.** Batching is a workflow capability, not only a hosted-service feature. In this notebook we will likely use the ALCHEMI Toolkit path, while BGR NIMs provide the deployable microservice route for the same kind of many-structure relaxation.
- <span style="color:#76b900"><strong>Toolkit</strong></span> **Custom workflows.** The ALCHEMI Toolkit provides Python building blocks for data structures, batching, wrappers, optimizers, integrators, and direct experimentation outside a hosted microservice.
- <span style="color:#00a3e0"><strong>Toolkit-Ops</strong></span> **GPU-accelerated kernels.** Toolkit-Ops supplies batched kernels for expensive common operations underneath atomistic simulation, including neighbor lists, dispersion corrections, and long-range electrostatics.
- **Workflow and audit.** The stack carries structures, constraints, runtime metadata, convergence status, caches, monitoring, and reference gates, because fast results are only useful if they remain traceable and reviewable.
- **Pluggable models.** The MLIP is the final computational engine chosen for the narrowed problem and execution path. In this notebook that engine is MACE-MPA-0 for periodic slab relaxation; other ALCHEMI workflows can use different potentials for different chemistry.

The discovery value is the complete loop: define a candidate set, generate valid structures, run batched GPU relaxations or simulations, rank reliable outputs, inspect failures, and decide which cases are ready for DFT, experiment, or domain-expert review.

![ALCHEMI Toolkit, Toolkit-Ops, and NIM architecture](assets/images/v0_core/alchemi_toolkit_community_ops.png)

## Why configuration search matters

![AdsorbML workflow](assets/images/v0_core/workflow_adsorbml_bgr.png)

This notebook uses surface adsorption as a concrete example of a broader discovery pattern: many plausible atomic structures must be generated, relaxed, ranked, and audited before a scientific conclusion is credible. Adsorption is a useful teaching case because the combinatorics are easy to see. A user can choose one plausible site and orientation, run one geometry optimization, and report the resulting binding energy, but that single-start workflow may only describe a local minimum.

AdsorbML showed why this matters for catalyst adsorption: different starting sites can relax into different local minima, and a single-start result can miss the lowest-energy configuration. The headline reliability numbers from Lan et al. (2023) must be checked against the exact paper/SI definitions before they are used as strict claims; here they motivate the workflow rather than serve as pair-level validation.

The broader lesson is that batched relaxation changes the scientific object being computed. A single relaxation asks, "What happens to this initial guess?" A batched search asks, "Across a controlled set of candidate structures, which relaxed outputs are reliable, which are lowest or most relevant, and which cases deserve stricter validation?" In this adsorption example, that becomes a direct test of whether a narrow single-starting-point protocol would have biased the reported binding mode or energy.

![Local minima from different starts](assets/images/v0_core/phenomenon_local_minima.png)

## The 9-pair panel

| Host | CO | H2O | CH3OH |
|---|---|---|---|
| Cu(111) fcc | top-site context | O-down tilted context | O-down context |
| Pd(111) fcc | hollow-site context | O-down tilted context | O-down context |
| alpha-Al2O3(0001) | Al-top context | Al-top O-down context | Al-top O-down + H-bond context |

These rows are chosen for methodological clarity and reference-density, not for novelty as catalyst discoveries. Reference rows currently labelled `context` are useful for interpretation, but they must not be used for strict apples-to-apples error statistics until slab model, coverage, functional, dispersion treatment, frozen layers, and sign convention are pinned in `references/manifest.yml`.

The expected scientific outcomes are therefore framed as review categories:

- **Validation candidate:** the batch minimum agrees with a well-supported reference site and falls inside the model-level uncertainty envelope once a strict or near-strict reference is available.
- **Configuration-search case:** a top-site-only or otherwise narrow search would have returned a higher-energy local minimum.
- **Review case:** the MACE-relaxed minimum, reference context, or geometry classification disagrees and requires dataset/literature review before interpretation.

## What the ALCHEMI BGR NIM provides

The BGR NIM is a containerised inference service wrapping a geometry-optimisation loop around an MLIP. The Docker Compose stack pulls `nvcr.io/nim/nvidia/alchemi-bgr:1.0.0`, configured for:

- **Model**: MACE-MPA-0 (MACE trained on MPtrj + sAlex)
- **Dispersion**: DFT-D3(BJ), enabled server-side via `ALCHEMI_NIM_DFT3_ENABLED=true`
- **Periodic**: `ALCHEMI_NIM_PBC=true` — all our slabs need it
- **Endpoint**: `POST /v1/infer` with a list of `BGRAtomicData` structures

Each request carries independent structures as a batch. In the NIM service route, `/v1/infer` handles the batch behind an HTTP API. In the Toolkit route used here, the same batching idea is exposed directly through `AtomicData` and `Batch`. The H2O section below demonstrates that batching mechanism before the larger adsorption panel.

Observability for the service route: Prometheus scrapes `/v1/metrics`; Grafana at `localhost:3000` shows live request and queue metrics while the NIM runs.


## Scope and quantitative uncertainty

Published MACE benchmark tables provide the uncertainty scale used in this tutorial. For MACE-MPA-0+D3 on the OC157 molecule-surface relative-energy task, the cached arXiv v3 supplement reports a 0.28 eV MAD. The older 0.42 eV / 121-of-157 numbers belong to an earlier arXiv/model-naming version and should not be used for this MPA-0 run. This is a screening-level uncertainty scale for configuration ranking; it is not chemical accuracy.

Interpreted conservatively, this scale means:

- The model-level error is much larger than chemical accuracy, so sub-0.1 eV conclusions are not justified from this notebook alone.
- The error is still useful for screening-level workflow demonstrations, especially when the question is whether a configuration search changes the apparent minimum by several tenths of an eV.
- Relative energies among configurations on the same surface may benefit from error cancellation, but that assumption should be tested against strict references where available.

What the notebook can support after execution:

- A controlled comparison of starting-site bias within each adsorbate-surface pair.
- A final-site classification based on relaxed geometry rather than initial labels.
- A reference-aware status table that keeps contextual literature separate from strict parity validation.

What it cannot support by itself:

- Final catalyst selection.
- Claims about activation barriers, coverage effects, electrochemical free energies, explicit solvent, magnetic chemistry, reducible defects, or finite-temperature entropic corrections.
- Strict MACE-vs-DFT error statistics until exact dataset or literature records are pinned.

## Start BGR before running the notebook

The BGR NIM should be started **outside** this notebook. Do not use a `%%bash` startup cell here: cell magics require an active Jupyter kernel, so they are the wrong bootstrap mechanism for bringing the kernel-side service up.

Workshop / cluster path:

```bash
cd part-1-nim
scripts/deploy.sh setup <login-host> <compute-node>
```

Local Docker path, if you are running the stack manually:

```bash
cd part-1-nim
docker compose up -d bgr
curl -sf http://localhost:8000/v1/health/ready
```

When finished, stop the local stack with `docker compose down`. The notebook cells below only check `BGR_SERVER` and call `/v1/infer`; they do not start containers.


---

## Control panel

Use this cell to choose how the rest of the notebook will run. Later cells read these same settings, so one change here is enough to switch between a quick teaching run and the full adsorption search.

The choices below are plain Python variables. Edit the values directly in the next cell; this is a tutorial notebook, not a production configuration file.

For a first pass, keep `SMALL_PANEL_MODE = True`. This runs one representative adsorbate-surface pair through the same setup, relaxation, ranking, and plotting steps, but with only a few starting structures. It is the fastest way to check that the notebook, backend, and visualization path are working.

When the workflow is behaving, set `SMALL_PANEL_MODE = False` to run the full tutorial panel. That mode is the one to use for the scientific comparison across all examples included here.

`BACKEND` chooses the execution path. In this version, keep `BACKEND = "toolkit"` so relaxations run through the ALCHEMI Toolkit on the cluster GPU. Use `BACKEND = "bgr_nim"` only for the service/API variant.

`USE_CACHED_RESPONSES` decides whether relaxation outputs are read from saved JSON instead of recomputed. Cached mode is useful when GPU or service access is unavailable, or when reviewing analysis cells. Set it to `False` for new calculations.


In [ ]:
import os
import sys

# === Run choices ===========================================================

# Start with the small panel while learning or checking the setup. It uses one
# representative pair but still exercises generation, relaxation, ranking, and plotting.
SMALL_PANEL_MODE = True

# Choose the execution path:
# - "toolkit" runs ALCHEMI Toolkit Python on the cluster GPU.
# - "bgr_nim" uses the BGR NIM service/API path.
BACKEND = "toolkit"

# Read saved relaxation outputs instead of recomputing them. Useful when GPU or
# service access is unavailable, or when reviewing downstream analysis cells.
USE_CACHED_RESPONSES = False

# Structure images are part of the evidence trail, not decoration. Keep this True
# for final tutorial outputs; set False only for energy-table debugging on a
# machine where OVITO/VisRTX cannot initialize.
REQUIRE_VISRTX_RENDER = True

# === Notebook housekeeping =================================================

# Keep model and library caches inside the repo so setup is repeatable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "part-1-nim" else os.getcwd()
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(REPO_ROOT, ".xdg-cache"))
os.environ.setdefault("TORCH_HOME", os.path.join(REPO_ROOT, ".torch-cache"))
os.environ.setdefault("HF_HOME", os.path.join(REPO_ROOT, ".hf-cache"))
os.environ.setdefault("WARP_CACHE_PATH", os.path.join(REPO_ROOT, ".warp-cache"))
os.environ.setdefault("MPLCONFIGDIR", os.path.join(REPO_ROOT, ".matplotlib-cache"))
os.environ.setdefault("TORCH_COMPILE_DISABLE", "1")
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")

# === Local paths ===========================================================
OUTPUT_DIR = "outputs"
CACHE_DIR = os.path.join("cached_responses", "adsorption-search")
ASSETS_DIR = "assets"
IMAGES_DIR = os.path.join(ASSETS_DIR, "images")
PLOTS_DIR = os.path.join(IMAGES_DIR, "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# === BGR NIM service settings =============================================
BGR_SERVER = "http://localhost:8000"

# BGR optimizer force tolerance in eV/A. None means: use the service preset.
OPTTOL = None

# === ALCHEMI Toolkit settings =============================================
TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = False
TOOLKIT_DT = 0.01
TOOLKIT_N_STEPS = 5000
TOOLKIT_FMAX = 0.05

# Dispersion is not part of the current Toolkit teaching run. Keep this explicit
# so later metadata does not imply a correction that was not used.
TOOLKIT_REQUIRE_D3BJ = False
TOOLKIT_D3BJ = None

# Use a smaller concurrency limit for the small run so its output is easier to inspect.
MAX_CONCURRENT_PAIRS = 3 if SMALL_PANEL_MODE else 9

print(f"SMALL_PANEL_MODE     : {SMALL_PANEL_MODE}")
print(f"BACKEND              : {BACKEND}")
print(f"USE_CACHED_RESPONSES : {USE_CACHED_RESPONSES}")
print(f"BGR_SERVER           : {BGR_SERVER}")
print(f"CACHE_DIR            : {CACHE_DIR}")
print(f"TOOLKIT_CHECKPOINT   : {TOOLKIT_CHECKPOINT}")
print(f"TOOLKIT_DEVICE       : {TOOLKIT_DEVICE}")
print(f"TOOLKIT_REQUIRE_D3BJ : {TOOLKIT_REQUIRE_D3BJ}")
print(f"TOOLKIT_D3BJ         : {'set' if TOOLKIT_D3BJ else 'not set'}")
print(f"REQUIRE_VISRTX_RENDER: {REQUIRE_VISRTX_RENDER}")
print(f"MAX_CONCURRENT_PAIRS : {MAX_CONCURRENT_PAIRS}")


In [ ]:
## Package versions and imports

In [ ]:
import sys
from importlib.metadata import version as _pkgver

print(f"Python     : {sys.version.split()[0]}")
for pkg in ("ase", "numpy", "pandas", "matplotlib", "pymatgen", "pydantic",
            "requests", "aiohttp", "ipywidgets", "ovito", "tqdm"):
    try:
        print(f"{pkg:<10} : {_pkgver(pkg)}")
    except Exception as e:
        print(f"{pkg:<10} : NOT INSTALLED ({type(e).__name__})")


In [ ]:
# TODO[CREATOR-CLEANUP]: remove this autoreload block before public tutorial export.
# It is only for notebook authors editing helpers/*.py during development.
%load_ext autoreload
%autoreload 2

import ase
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ase.build import molecule as ase_molecule

from helpers import (
    # BGR client + cache
    check_endpoint,
    run_bgr_or_load_cache,
    async_run_bgr_or_load_cache,
    # Explicit relaxation backend selection
    BackendUnavailableError, RelaxationBackendConfig, ToolkitD3BJConfig,
    check_toolkit_native_api, get_relaxation_backend,
    # Data models
    BGRAtomicData, BGRReply, OptimizationResult,
    ase_to_atomic_data, atomic_data_to_ase,
    # Host builders (AdsorbML panel)
    build_cu111_slab, build_pd111_slab,
    build_alpha_alumina_0001_slab,
    # Configuration grid
    Configuration, build_config_grid,
    build_co, build_h2o, build_methanol,
    ADSORBATE_ORIENTATIONS, sites_for_host,
    # Slab helpers
    make_active_mask, find_central_site,
    compute_adsorption_energy,
    # Apples-to-apples analysis
    ADSORPTION_ENERGY_FORMULA,
    build_pair_results_table, summarize_pair_validation,
    strict_parity_subset,
    # Visualisation
    render_structure_ovito, create_interactive_view, display_widgets_row,
    display_inline, structure_summary_table,
    # Throughput scan
    measure_batch_throughput, sweep_batch_throughput, plot_throughput,
    # References + MAD constants
    ADSORBML_REFERENCES, get_adsorbml_reference,
    MACE_MPA0_OC157_MAD_EV, MACE_MP0B3_OC157_MAD_EV,
    # Constants
    KJ_MOL_TO_EV, EV_TO_KJ_MOL,
)
print("helpers imported OK")


## Backend check and runtime metadata

Confirm the selected execution path before running calculations. For the Toolkit path, this checks that the native Toolkit API is available in the kernel environment. For the BGR NIM path, it checks the service endpoint and prints service metadata when available.


In [ ]:
import requests

if BACKEND not in {"bgr_nim", "toolkit"}:
    raise ValueError(f"BACKEND must be 'bgr_nim' or 'toolkit', got {BACKEND!r}")

BGR_LIVE = check_endpoint(BGR_SERVER) if (BACKEND == "bgr_nim" and not USE_CACHED_RESPONSES) else False
cache_files = [name for name in os.listdir(CACHE_DIR) if name.endswith(".json")]

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

if USE_CACHED_RESPONSES:
    print(f"Using cached relaxation responses: {len(cache_files)} file(s) in {CACHE_DIR}")
elif BACKEND == "bgr_nim":
    print(f"BGR endpoint live: {BGR_LIVE}")
    if BGR_LIVE:
        for path in ("/v1/metadata", "/v1/status", "/v1/models"):
            try:
                r = requests.get(BGR_SERVER + path, timeout=5)
                if r.ok and r.headers.get("content-type", "").startswith("application/json"):
                    meta = r.json()
                    print(f"GET {path}:")
                    for k, v in (meta.items() if isinstance(meta, dict) else []):
                        print(f"  {k}: {v}")
                    break
            except requests.RequestException:
                continue
        else:
            print("No metadata endpoint responded; check service logs for the deployed model.")
    else:
        print("BGR endpoint is down; cached responses will be used if available.")

    print(f"Cached relaxation responses: {len(cache_files)} file(s) in {CACHE_DIR}")
    if not BGR_LIVE and not cache_files:
        raise RuntimeError(
            "No live BGR endpoint and no cached relaxation responses are available. "
            "Start the BGR NIM service or provide cached JSON responses under "
            "cached_responses/adsorption-search/."
        )
elif BACKEND == "toolkit":
    status = check_toolkit_native_api()
    print(status["message"])
    if TOOLKIT_REQUIRE_D3BJ and TOOLKIT_D3BJ is None:
        print("Toolkit D3(BJ) is required but not configured; backend construction will fail until TOOLKIT_D3BJ is set.")
    elif not TOOLKIT_REQUIRE_D3BJ and TOOLKIT_D3BJ is None:
        print("Toolkit run is explicit MACE-only: D3(BJ) is disabled.")

RELAXATION_BACKEND = get_relaxation_backend(RelaxationBackendConfig(
    name=BACKEND,
    cache_dir=CACHE_DIR,
    use_cached_responses=USE_CACHED_RESPONSES,
    timeout=1800,
    opttol=OPTTOL,
    bgr_server=BGR_SERVER,
    bgr_endpoint_live=BGR_LIVE,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_REQUIRE_D3BJ,
))
print(f"Relaxation backend ready: {RELAXATION_BACKEND.name}")


---

## Batched H2O relaxation: the smallest speedup example

Before building surfaces, start with a controlled molecule-only example. The scientific object is deliberately simple: many independent water structures, each in a vacuum box, relaxed in one batched Toolkit workflow.

This is not a claim about water chemistry. It is a mechanical demonstration of why batching matters: the model is loaded once, structures are packed into one `Batch`, and the GPU works on many small systems together instead of repeating the same setup one structure at a time.


### One figure: H2O examples and batch speedup

The next cell uses the official Toolkit objects directly: `AtomicData.from_atoms`, `Batch.from_data_list`, `MACEWrapper.from_checkpoint`, `ConvergenceHook`, and `FIRE2`. It records the gas-phase H2O energy needed later and saves one NVIDIA-styled figure that combines example structures with the measured speedup.

In small-panel mode the sweep stays short. In the full run it tests larger H2O batches before moving on to the adsorption panel.


In [ ]:
from time import perf_counter


def gas_phase_h2o_atoms(box: float = 15.0, rotation_deg: float = 0.0) -> ase.Atoms:
    """A single water molecule centered in a cubic vacuum box."""
    h2o = ase_molecule("H2O")
    h2o.rotate(rotation_deg, "z", center="COM")
    h2o.rotate(rotation_deg / 3.0, "x", center="COM")
    h2o.set_cell([box, box, box])
    h2o.set_pbc(True)
    h2o.center()
    return h2o


def make_h2o_examples(count: int) -> list[ase.Atoms]:
    return [gas_phase_h2o_atoms(rotation_deg=17.0 * i) for i in range(count)]


def plot_h2o_batch_speedup(speedup_df: pd.DataFrame, examples: list[ase.Atoms], output_path: str) -> None:
    nv_green = "#76B900"
    nv_blue = "#00A3E0"
    dark = "#101820"
    light = "#F3F5F7"
    muted = "#A8B0B8"
    oxygen = "#E53935"
    hydrogen = "#E6EDF3"

    fig = plt.figure(figsize=(10.8, 4.6), facecolor=dark)
    grid = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.55], wspace=0.28)

    ax_mol = fig.add_subplot(grid[0, 0], projection="3d", facecolor=dark)
    offsets = np.linspace(-4.2, 4.2, len(examples)) if examples else []
    for offset, atoms in zip(offsets, examples):
        positions = atoms.positions - atoms.get_center_of_mass() + np.array([offset, 0.0, 0.0])
        symbols = atoms.get_chemical_symbols()
        for i, (symbol, pos) in enumerate(zip(symbols, positions)):
            color = oxygen if symbol == "O" else hydrogen
            size = 210 if symbol == "O" else 95
            ax_mol.scatter(pos[0], pos[1], pos[2], s=size, color=color, edgecolor=dark, linewidth=0.7)
            for j in range(i):
                if np.linalg.norm(positions[i] - positions[j]) < 1.25:
                    xs, ys, zs = zip(positions[i], positions[j])
                    ax_mol.plot(xs, ys, zs, color=muted, linewidth=2.0, alpha=0.9)
    ax_mol.set_title("Independent H2O structures", color=light, pad=14, fontsize=12)
    ax_mol.set_axis_off()
    ax_mol.view_init(elev=18, azim=-62)
    ax_mol.set_xlim(-5.5, 5.5)
    ax_mol.set_ylim(-2.0, 2.0)
    ax_mol.set_zlim(-2.0, 2.0)

    ax = fig.add_subplot(grid[0, 1], facecolor=dark)
    df = speedup_df.sort_values("batch_size")
    ax.plot(df["batch_size"], df["speedup_vs_single"], color=nv_green, marker="o", linewidth=2.6, label="measured")
    ax.plot(df["batch_size"], df["batch_size"], color=muted, linestyle="--", linewidth=1.2, label="ideal linear")
    ax.fill_between(df["batch_size"], df["speedup_vs_single"], color=nv_green, alpha=0.16)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("H2O structures in one Toolkit batch", color=light)
    ax.set_ylabel("speedup vs one-at-a-time", color=light)
    ax.set_title("Batching amortizes fixed work", color=light, pad=12, fontsize=12)
    ax.tick_params(colors=light)
    for spine in ax.spines.values():
        spine.set_color("#4B5563")
    ax.grid(True, color="#2F3A44", linewidth=0.8, alpha=0.8)
    ax.legend(facecolor=dark, edgecolor="#4B5563", labelcolor=light, loc="upper left")

    last = df.iloc[-1]
    ax.annotate(
        f"{last['speedup_vs_single']:.1f}x",
        xy=(last["batch_size"], last["speedup_vs_single"]),
        xytext=(-44, 18),
        textcoords="offset points",
        color=nv_blue,
        fontsize=12,
        arrowprops={"arrowstyle": "->", "color": nv_blue, "lw": 1.4},
    )

    fig.suptitle("ALCHEMI Toolkit batched H2O relaxation", color=light, fontsize=15, y=0.98)
    fig.savefig(output_path, dpi=170, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)


H2O_BATCH_SIZES = [1, 2, 4, 8, 16] if SMALL_PANEL_MODE else [1, 2, 4, 8, 16, 32, 64]
H2O_BENCH_STEPS = 20 if SMALL_PANEL_MODE else 80
H2O_SPEEDUP_CACHE = os.path.join(CACHE_DIR, "h2o_toolkit_batch_speedup.csv")
H2O_SPEEDUP_FIGURE = os.path.join(PLOTS_DIR, "h2o_toolkit_batch_speedup.png")

if USE_CACHED_RESPONSES:
    if not os.path.exists(H2O_SPEEDUP_CACHE):
        raise RuntimeError(f"Cached H2O speedup table not found: {H2O_SPEEDUP_CACHE}")
    h2o_speedup_df = pd.read_csv(H2O_SPEEDUP_CACHE)
else:
    if BACKEND != "toolkit":
        raise RuntimeError("This section uses the ALCHEMI Toolkit API directly; set BACKEND = 'toolkit'.")

    import torch
    from nvalchemi.data import AtomicData, Batch
    from nvalchemi.dynamics import ConvergenceHook
    from nvalchemi.dynamics.hooks import NaNDetectorHook
    from nvalchemi.dynamics.optimizers import FIRE2
    from nvalchemi.models.mace import MACEWrapper

    if TOOLKIT_DEVICE == "auto":
        if not torch.cuda.is_available():
            raise RuntimeError("Toolkit speedup example needs a CUDA GPU. Use cached responses when GPU access is unavailable.")
        toolkit_device = torch.device("cuda")
    else:
        toolkit_device = torch.device(TOOLKIT_DEVICE)
    toolkit_dtype = getattr(torch, TOOLKIT_DTYPE)

    h2o_model = MACEWrapper.from_checkpoint(
        TOOLKIT_CHECKPOINT,
        device=toolkit_device,
        dtype=toolkit_dtype,
        enable_cueq=TOOLKIT_ENABLE_CUEQ,
        compile_model=TOOLKIT_COMPILE_MODEL,
    )
    h2o_model.model_config.active_outputs = {"energy", "forces"}

    def run_h2o_toolkit_batch(batch_size: int) -> dict[str, float | int]:
        atoms_list = make_h2o_examples(batch_size)
        atomic_data = [AtomicData.from_atoms(atoms, device=toolkit_device, dtype=toolkit_dtype) for atoms in atoms_list]
        batch = Batch.from_data_list(atomic_data, device=toolkit_device)
        optimizer = FIRE2(
            h2o_model,
            dt=TOOLKIT_DT,
            n_steps=H2O_BENCH_STEPS,
            convergence_hook=ConvergenceHook.from_fmax(threshold=TOOLKIT_FMAX, source_status=0, target_status=1),
        )
        for hook in h2o_model.make_neighbor_hooks():
            optimizer.register_hook(hook)
        optimizer.register_hook(NaNDetectorHook())

        if str(toolkit_device).startswith("cuda"):
            torch.cuda.synchronize(toolkit_device)
        start = perf_counter()
        relaxed_batch = optimizer.run(batch)
        if str(toolkit_device).startswith("cuda"):
            torch.cuda.synchronize(toolkit_device)
        wall_time = perf_counter() - start

        energies = []
        for idx in range(batch_size):
            data = relaxed_batch.get_data(idx)
            energies.append(float(data.energy.detach().cpu().reshape(-1)[0]))
        return {
            "batch_size": batch_size,
            "wall_time_s": wall_time,
            "structures_per_s": batch_size / wall_time,
            "energy_mean_eV": float(np.mean(energies)),
            "energy_std_eV": float(np.std(energies)),
        }

    h2o_speedup_rows = [run_h2o_toolkit_batch(size) for size in H2O_BATCH_SIZES]
    h2o_speedup_df = pd.DataFrame(h2o_speedup_rows)
    h2o_speedup_df.to_csv(H2O_SPEEDUP_CACHE, index=False)

single_time = float(h2o_speedup_df.loc[h2o_speedup_df["batch_size"] == 1, "wall_time_s"].iloc[0])
h2o_speedup_df["one_at_a_time_s"] = single_time * h2o_speedup_df["batch_size"]
h2o_speedup_df["speedup_vs_single"] = h2o_speedup_df["one_at_a_time_s"] / h2o_speedup_df["wall_time_s"]
E_H2O_gas = float(h2o_speedup_df.loc[h2o_speedup_df["batch_size"] == 1, "energy_mean_eV"].iloc[0])
results = h2o_speedup_df.to_dict("records")

plot_h2o_batch_speedup(h2o_speedup_df, make_h2o_examples(min(4, len(H2O_BATCH_SIZES))), H2O_SPEEDUP_FIGURE)
display_inline(H2O_SPEEDUP_FIGURE)
print(f"E(H2O, gas) = {E_H2O_gas:.4f} eV")
print(f"Saved: {os.path.abspath(H2O_SPEEDUP_FIGURE)}")
h2o_speedup_df


## Official Toolkit API mini-guide

The clean boundary is `ASE Atoms -> AtomicData -> Batch`. ASE is the chemistry authoring layer; Toolkit begins when we call `AtomicData.from_atoms`. The runnable cell below repeats the pattern on four water structures using the public Toolkit API directly.

```python
atomic_data = [AtomicData.from_atoms(atoms, device=device, dtype=dtype) for atoms in atoms_list]
batch = Batch.from_data_list(atomic_data, device=device)
model = MACEWrapper.from_checkpoint(checkpoint, device=device, dtype=dtype)
optimizer = FIRE2(model, dt=0.01, n_steps=20, convergence_hook=ConvergenceHook.from_fmax(0.05))
relaxed_batch = optimizer.run(batch)
```


In [ ]:
if USE_CACHED_RESPONSES:
    raise RuntimeError("This mini-guide cell is a live Toolkit API exercise; set USE_CACHED_RESPONSES = False to run it.")

import torch
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import ConvergenceHook
from nvalchemi.dynamics.hooks import NaNDetectorHook
from nvalchemi.dynamics.optimizers import FIRE2
from nvalchemi.models.mace import MACEWrapper

# Run the same official Toolkit pattern on a tiny batch.
demo_atoms_list = make_h2o_examples(4)

demo_device = toolkit_device if "toolkit_device" in globals() else torch.device("cuda")
demo_dtype = toolkit_dtype if "toolkit_dtype" in globals() else getattr(torch, TOOLKIT_DTYPE)
demo_model = h2o_model if "h2o_model" in globals() else MACEWrapper.from_checkpoint(
    TOOLKIT_CHECKPOINT,
    device=demo_device,
    dtype=demo_dtype,
    enable_cueq=TOOLKIT_ENABLE_CUEQ,
    compile_model=TOOLKIT_COMPILE_MODEL,
)
demo_model.model_config.active_outputs = {"energy", "forces"}

demo_atomic_data = [AtomicData.from_atoms(atoms, device=demo_device, dtype=demo_dtype) for atoms in demo_atoms_list]
demo_batch = Batch.from_data_list(demo_atomic_data, device=demo_device)

demo_optimizer = FIRE2(
    demo_model,
    dt=TOOLKIT_DT,
    n_steps=min(20, H2O_BENCH_STEPS),
    convergence_hook=ConvergenceHook.from_fmax(threshold=TOOLKIT_FMAX, source_status=0, target_status=1),
)
for hook in demo_model.make_neighbor_hooks():
    demo_optimizer.register_hook(hook)
demo_optimizer.register_hook(NaNDetectorHook())

if str(demo_device).startswith("cuda"):
    torch.cuda.synchronize(demo_device)
demo_relaxed_batch = demo_optimizer.run(demo_batch)
if str(demo_device).startswith("cuda"):
    torch.cuda.synchronize(demo_device)

print(f"Toolkit batch contains {getattr(demo_batch, 'num_graphs', len(demo_atoms_list))} H2O structures")
print(type(demo_batch).__name__, "->", type(demo_relaxed_batch).__name__)


In [ ]:
demo_rows = []
for idx, atoms in enumerate(demo_atoms_list):
    relaxed = demo_relaxed_batch.get_data(idx)
    energy = float(relaxed.energy.detach().cpu().reshape(-1)[0])
    forces = relaxed.forces.detach().cpu().numpy().reshape(-1, 3)
    demo_rows.append({
        "structure": f"H2O_{idx}",
        "atoms": len(atoms),
        "energy_eV": energy,
        "fmax_eV_A": float(np.linalg.norm(forces, axis=1).max()),
    })

demo_api_df = pd.DataFrame(demo_rows)
assert len(demo_api_df) == len(demo_atoms_list)
demo_api_df


**Reading this section:** the H2O batch is a runtime sanity example. It shows the batching mechanism and records the gas-phase H2O reference used later, but it is not the scientific adsorption example. The adsorption panel below applies the same batching idea to chemically meaningful surface-starting configurations.


---

## Build the 3-host slab panel

Three closed-shell, non-magnetic surfaces, all with dense OC20/OC22 and surface-science reference data: **Cu(111)**, **Pd(111)**, **α-Al₂O₃(0001)**. OC20 convention: 4-layer slabs, bottom 2 layers frozen during relaxation, 15 Å vacuum (inside MACE's 12 Å receptive-field cutoff plus slack).


In [ ]:
HOSTS = {
    "Cu(111)":     build_cu111_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(3, 3, 1)),
    "Pd(111)":     build_pd111_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(3, 3, 1)),
    # 2x2 alumina avoids the artificial high coverage of a 1x1 oxide slab.
    "Al2O3(0001)": build_alpha_alumina_0001_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
}
HOST_NAMES = list(HOSTS.keys())

for name, atoms in HOSTS.items():
    symbols = atoms.get_chemical_symbols()
    comp = {s: symbols.count(s) for s in sorted(set(symbols))}
    formula = " ".join(f"{s}{n}" for s, n in comp.items())
    print(f"  {name:<14}  atoms={len(atoms):>4}  cell={atoms.cell.lengths().round(2).tolist()}  {formula}")


### Interactive 3-D views of the three hosts

In [ ]:
display_widgets_row(
    [(name, atoms) for name, atoms in HOSTS.items()],
    width="300px", height="260px",
)


## Build the 3 adsorbates

**CO** (C-down and O-down orientations), **H₂O** (O-down, H-down, flat), **CH₃OH** (O-down, methyl-down). Geometries come from `ase.build.molecule` where available, matching experimental bond lengths.


In [ ]:
ADSORBATES = ["CO", "H2O", "CH3OH"]

sample_adsorbates = []
for name in ADSORBATES:
    for orient in ADSORBATE_ORIENTATIONS[name][:1]:  # show first orientation
        ads = {"CO": build_co, "H2O": build_h2o, "CH3OH": build_methanol}[name](orient)
        # give each a box for visualisation
        ads.set_cell([8, 8, 8]); ads.pbc = True; ads.center()
        sample_adsorbates.append((f"{name} ({orient})", ads))

display_widgets_row(sample_adsorbates, width="240px", height="220px")


## Clean-slab relaxation

Three concurrent `/v1/infer` calls, one per host, with `tqdm` per-host progress. Bottom-half `active_mask` freezes the two lower layers (standard OC20 convention). Each host gets its own cache file `clean_<host>.json`.


In [ ]:
import asyncio, aiohttp
from tqdm.auto import tqdm


def _safe(name: str) -> str:
    return name.replace("(", "_").replace(")", "").replace(",", "_").replace("/", "_")


async def _relax_clean(name, atoms, session):
    mask = make_active_mask(atoms, bottom_fraction=0.5)
    data = ase_to_atomic_data(atoms, structure_id=f"clean_{_safe(name)}", active_mask=mask)
    reply = await RELAXATION_BACKEND.async_relax(
        [data], label=f"clean_{_safe(name)}", cellopt=False, session=session,
    )
    return name, reply.atoms[0]


async def _relax_all_clean():
    connector = aiohttp.TCPConnector(limit=len(HOST_NAMES))
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [asyncio.create_task(_relax_clean(n, HOSTS[n], session)) for n in HOST_NAMES]
        out = {}
        with tqdm(total=len(tasks), desc="Relaxing clean slabs", ncols=80) as pbar:
            for coro in asyncio.as_completed(tasks):
                name, opt = await coro
                out[name] = opt
                pbar.set_postfix_str(f"done: {name}")
                pbar.update(1)
    return [out[n] for n in HOST_NAMES]


clean_opts = await _relax_all_clean()

rows, E_HOST, HOST_RELAXED = [], {}, {}
for name, opt in zip(HOST_NAMES, clean_opts):
    relaxed = atomic_data_to_ase(opt)
    HOST_RELAXED[name] = relaxed
    E_HOST[name] = float(opt.energy)
    fmax = float(np.max(np.linalg.norm(np.array(opt.forces).reshape(-1, 3), axis=1)))
    rows.append({
        "Host": name, "atoms": len(relaxed),
        "converged": opt.converged, "n_steps": opt.num_optimization_steps,
        "max |F| (eV/A)": round(fmax, 4), "E_host (eV)": round(E_HOST[name], 4),
    })
pd.DataFrame(rows)


---

## Generate the AdsorbML configuration grid

For every (host × adsorbate) pair, enumerate starting configurations: sites × orientations × rotations × heights. In `SMALL_PANEL_MODE` we run only CO on Cu(111) with 4 starts; in full-panel mode, all selected host × adsorbate pairs are relaxed.


In [ ]:
def _env_list(name: str, default: list[str] | None = None) -> list[str] | None:
    value = os.environ.get(name)
    if value is None:
        return default
    value = value.strip()
    if value.lower() in {"", "none", "all", "*"}:
        return None
    return [item.strip() for item in value.split(",") if item.strip()]


def _env_float_tuple(name: str, default: tuple[float, ...]) -> tuple[float, ...]:
    value = os.environ.get(name)
    if value is None or value.strip() == "":
        return default
    return tuple(float(item.strip()) for item in value.split(",") if item.strip())


if SMALL_PANEL_MODE:
    PAIRS = [("Cu(111)", "CO")]
    # Default keeps the notebook quick. For scientific reruns, pass for example:
    # SMALL_PANEL_SITES=top,bridge,fcc,hcp SMALL_PANEL_HEIGHTS=1.6,1.8,2.0,2.2,2.4
    SITES_FILTER = _env_list("SMALL_PANEL_SITES", ["top", "bridge"])
    ORIENT_FILTER = _env_list("SMALL_PANEL_ORIENTATIONS", None)
    ROTATIONS = _env_float_tuple("SMALL_PANEL_ROTATIONS", (0.0, 60.0))
    HEIGHTS = _env_float_tuple("SMALL_PANEL_HEIGHTS", (2.2,))
else:
    PAIRS = [(h, a) for h in HOST_NAMES for a in ADSORBATES]
    SITES_FILTER = _env_list("FULL_PANEL_SITES", None)
    ORIENT_FILTER = _env_list("FULL_PANEL_ORIENTATIONS", None)
    ROTATIONS = _env_float_tuple("FULL_PANEL_ROTATIONS", (0.0, 60.0, 120.0))
    HEIGHTS = _env_float_tuple("FULL_PANEL_HEIGHTS", (2.2,))

GRID: dict[tuple[str, str], list[Configuration]] = {}
for host, adsorbate in PAIRS:
    GRID[(host, adsorbate)] = build_config_grid(
        host_name=host,
        slab=HOST_RELAXED[host],
        adsorbate_name=adsorbate,
        sites_filter=SITES_FILTER,
        orientations_filter=ORIENT_FILTER,
        rotations_deg=ROTATIONS,
        heights_A=HEIGHTS,
        frozen_fraction=0.5,
    )

total = sum(len(v) for v in GRID.values())
print(f"{'SMALL PANEL' if SMALL_PANEL_MODE else 'FULL PANEL'} mode: {len(PAIRS)} pair(s), "
      f"{total} total starting configurations")
print(f"Sites filter       : {SITES_FILTER if SITES_FILTER is not None else 'all'}")
print(f"Orientations filter: {ORIENT_FILTER if ORIENT_FILTER is not None else 'all'}")
print(f"Rotations (deg)    : {ROTATIONS}")
print(f"Heights (A)        : {HEIGHTS}")
for k, v in GRID.items():
    print(f"  {k[0]:<14} {k[1]:<6}  {len(v):>3} configs")


### Visualise a sample of starting configurations

One representative per pair (lowest site, first orientation, rot=0).


In [ ]:
sample = []
for (host, adsorbate), configs in GRID.items():
    if configs:
        sample.append((f"{adsorbate}/{host}", configs[0].atoms))
display_widgets_row(sample, width="280px", height="240px")


## Batch relaxation of every starting configuration

One BGR call per pair (all starts for that pair in one batch); pairs dispatched concurrently with `asyncio.gather`. Per-pair tqdm bar, per-pair cache `configs_<adsorbate>_<host>.json`.


In [ ]:
async def _relax_pair(host, adsorbate, configs, session):
    data_list = [
        ase_to_atomic_data(c.atoms, structure_id=c.label, active_mask=c.active_mask)
        for c in configs
    ]
    label = f"configs_{adsorbate}_{_safe(host)}"
    reply = await RELAXATION_BACKEND.async_relax(
        data_list, label=label, cellopt=False, session=session,
    )
    return (host, adsorbate), reply


async def _relax_all_pairs():
    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT_PAIRS)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [
            asyncio.create_task(_relax_pair(h, a, GRID[(h, a)], session))
            for h, a in PAIRS
        ]
        out = {}
        with tqdm(total=len(tasks), desc="Relaxing config grids", ncols=80) as pbar:
            for coro in asyncio.as_completed(tasks):
                key, reply = await coro
                out[key] = reply
                pbar.set_postfix_str(f"done: {key[1]}/{key[0]}")
                pbar.update(1)
    return out


pair_replies = await _relax_all_pairs()
print(f"Relaxed {sum(len(r.atoms) for r in pair_replies.values())} "
      f"configurations across {len(pair_replies)} pair(s).")


## Per-pair binding-energy distribution

For each pair, plot the relaxed binding energy of every starting configuration, marker shape by site, colour by orientation. The spread quantifies how much bias a single-starting-point protocol would have introduced on this pair.


In [ ]:
# Collect per-pair E_bind using the tutorial-wide apples-to-apples convention.
print(ADSORPTION_ENERGY_FORMULA)
E_ADS_GAS = {"H2O": E_H2O_gas}  # CO, CH3OH gas references computed below if needed

# Get gas references for CO and CH3OH once
for ads_name, builder in [("CO", build_co), ("CH3OH", build_methanol)]:
    if ads_name in {a for _, a in PAIRS} and ads_name not in E_ADS_GAS:
        mol = builder(ADSORBATE_ORIENTATIONS[ads_name][0])
        mol.set_cell([15, 15, 15]); mol.set_pbc(True); mol.center()
        r = RELAXATION_BACKEND.relax(
            [ase_to_atomic_data(mol, structure_id=f"gas_{ads_name}")],
            label=f"gas_{ads_name}",
        )
        E_ADS_GAS[ads_name] = float(r.atoms[0].energy)
        print(f"E({ads_name}, gas) = {E_ADS_GAS[ads_name]:.4f} eV")


PAIR_RESULTS: dict[tuple[str, str], pd.DataFrame] = {}
for (host, adsorbate), reply in pair_replies.items():
    PAIR_RESULTS[(host, adsorbate)] = build_pair_results_table(
        host=host,
        adsorbate=adsorbate,
        configs=GRID[(host, adsorbate)],
        opt_results=reply.atoms,
        clean_slab_atoms=HOST_RELAXED[host],
        e_clean_slab_ev=E_HOST[host],
        e_gas_ads_ev=E_ADS_GAS[adsorbate],
        backend=BACKEND,
    )

# Summary
pd.concat([df.assign(pair=f"{a}/{h}") for (h, a), df in PAIR_RESULTS.items()])\
  .pivot_table(index="pair", values="E_bind (eV)", aggfunc=["min", "median", "max", "count"])\
  .round(3)


In [ ]:
n_pairs = len(PAIR_RESULTS)
fig, axes = plt.subplots(n_pairs, 1, figsize=(9, max(2.5 * n_pairs, 3)), sharex=False)
if n_pairs == 1:
    axes = [axes]
site_markers = {"top": "o", "bridge": "s", "fcc": "^", "hcp": "v",
                "al-top": "o", "o-top": "D", "hollow": "P"}

for ax, ((host, adsorbate), df) in zip(axes, PAIR_RESULTS.items()):
    for site in df["final_site"].unique():
        sub = df[df["final_site"] == site]
        ax.scatter(sub["E_bind (eV)"], [0] * len(sub),
                   marker=site_markers.get(site, "o"), s=80, alpha=0.7,
                   label=f"final {site}", edgecolor="black", linewidth=0.5)
    emin = df["E_bind (eV)"].min()
    ax.axvline(emin, color="black", ls="--", lw=1, label=f"min = {emin:.3f} eV")
    # Reference/context line. Context rows are not parity references.
    ref = get_adsorbml_reference(host, adsorbate)
    if ref is not None and ref.E_bind_eV is not None:
        if ref.strict_for_parity:
            ax.axvline(ref.E_bind_eV, color="red", ls=":", lw=1.5,
                       label=f"parity ref {ref.E_bind_eV:.2f} eV ({ref.binding_site})")
        else:
            ax.axvline(ref.E_bind_eV, color="#777777", ls=":", lw=1.0,
                       label=f"context only {ref.E_bind_eV:.2f} eV ({ref.binding_site})")
    ax.set_yticks([])
    ax.set_xlabel("E_bind (eV)")
    ax.set_title(f"{adsorbate} on {host}  ·  {len(df)} starts  ·  spread = {df['E_bind (eV)'].max() - emin:.3f} eV",
                 fontsize=10)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
    ax.grid(True, axis="x", ls="--", alpha=0.4)

fig.tight_layout()
dist_path = os.path.join(PLOTS_DIR, "binding_distribution.png")
fig.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display_inline(dist_path)
print(f"Saved: {os.path.abspath(dist_path)}")


### Batch-minimum before/after views

For each pair, show the starting structure that produced the batch minimum next to its final relaxed structure. This is the visual counterpart to the energy distribution: the minimum is a relaxed geometry, not just the initial site label.


In [ ]:
winner_items = []
for (host, adsorbate), df in PAIR_RESULTS.items():
    idx = int(df["E_bind (eV)"].idxmin())
    start_atoms = GRID[(host, adsorbate)][idx].atoms
    final_atoms = atomic_data_to_ase(pair_replies[(host, adsorbate)].atoms[idx])
    e = df.loc[idx, "E_bind (eV)"]
    site = df.loc[idx, "final_site"]
    winner_items.extend([
        (f"start {adsorbate}/{host}", start_atoms),
        (f"final {site}, {e:.2f} eV", final_atoms),
    ])

display_widgets_row(winner_items, width="260px", height="230px", show_cell=False)


### Publication-quality OVITO renders of batch minima

The interactive widgets above are for inspection. The saved final-structure images below are the important result renders, so they explicitly request OVITO's VisRTX/ANARI renderer rather than relying on the default renderer. If VisRTX is unavailable on the machine, this cell should fail visibly instead of silently producing lower-fidelity result images.


In [ ]:
IMPORTANT_RENDER_DIR = os.path.join(OUTPUT_DIR, "ovito_visrtx")
os.makedirs(IMPORTANT_RENDER_DIR, exist_ok=True)

visrtx_render_paths = []
visrtx_render_failures = []
for (host, adsorbate), df in PAIR_RESULTS.items():
    idx = int(df["E_bind (eV)"].idxmin())
    final_atoms = atomic_data_to_ase(pair_replies[(host, adsorbate)].atoms[idx])
    e = df.loc[idx, "E_bind (eV)"]
    site = df.loc[idx, "final_site"]
    path = os.path.join(
        IMPORTANT_RENDER_DIR,
        f"winner_{_safe(adsorbate)}_{_safe(host)}_{site}.png",
    )
    try:
        render_structure_ovito(
            final_atoms,
            output_path=path,
            size=(1200, 900),
            renderer="visrtx",
            samples_per_pixel=64,
            show_cell=False,
        )
    except RuntimeError as exc:
        visrtx_render_failures.append({
            "path": path,
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        if REQUIRE_VISRTX_RENDER:
            raise RuntimeError(
                "OVITO VisRTX/ANARI render was required for an important result, "
                "but renderer='visrtx' failed. Set REQUIRE_VISRTX_RENDER=0 only "
                "for an explicit energy-only run on a machine without a working "
                "VisRTX device."
            ) from exc
    else:
        visrtx_render_paths.append(path)

for path in visrtx_render_paths:
    display_inline(path)
if visrtx_render_paths:
    print("Saved VisRTX renders:")
    for path in visrtx_render_paths:
        print(f"  {os.path.abspath(path)}")
if visrtx_render_failures:
    print("VisRTX render failures:")
    for failure in visrtx_render_failures:
        print(f"  {failure['path']}: {failure['error_type']}: {failure['error']}")


## Global-minimum site identification vs published reference

For each pair, pick the batch-minimum configuration and compare against the current reference row. Rows labelled `context` are useful site/energy checkpoints, but not strict apples-to-apples parity data until the exact OC20/OC22/AdsorbML records are pinned.

- **Site match** — does the MACE-lowest config sit at the published global-min site?
- **Delta E vs reference** — in eV, and in units of the MACE-MPA-0+D3 OC157 molecule-surface relative-energy MAD (0.28 eV), only interpreted statistically for strict / near-strict rows.

In [ ]:
summary_df = summarize_pair_validation(PAIR_RESULTS, ADSORBML_REFERENCES)

display_cols = [
    "pair", "tier", "reference_scope", "status", "site_match",
    "MACE_site", "reference_site", "E_MACE_eV", "E_ref_eV",
    "delta_E_eV", "abs_delta_over_MAD",
]
summary_df[display_cols]


### Strict parity eligibility

The table above separates **strict / near-strict** references from **context** references. Strict parity requires matching slab model, coverage, functional, dispersion convention, and sign convention. Context references can guide interpretation, but they are not used for apples-to-apples error statistics.


In [ ]:
strict_df = strict_parity_subset(summary_df)
if strict_df.empty:
    print("No strict apples-to-apples DFT rows are enabled yet.")
    print("Current reference energies are labelled context until exact OC20/OC22/AdsorbML rows are cached with matching slab/coverage/method metadata.")
else:
    strict_df


### Site-agreement heatmap

A quick visual check of whether the MACE batch minimum lands on the published reference site. This uses the final relaxed geometry, not the starting label.


In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

site_df = summary_df.copy()
site_df["site_score"] = site_df["site_match"].map({True: 1, False: -1}).fillna(0)
heat = site_df.pivot(index="host", columns="adsorbate", values="site_score")
heat = heat.reindex(HOST_NAMES).reindex(columns=ADSORBATES)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
cmap = ListedColormap(["#d62728", "#bdbdbd", "#2ca02c"])
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)
im = ax.imshow(heat.values.astype(float), cmap=cmap, norm=norm)
ax.set_xticks(range(len(ADSORBATES))); ax.set_xticklabels(ADSORBATES)
ax.set_yticks(range(len(HOST_NAMES))); ax.set_yticklabels(HOST_NAMES)
for i, host in enumerate(HOST_NAMES):
    for j, ads in enumerate(ADSORBATES):
        row = site_df[(site_df["host"] == host) & (site_df["adsorbate"] == ads)]
        if len(row):
            value = row.iloc[0]["site_match"]
            text = "match" if value == True else "diff" if value == False else "n/a"
            ax.text(j, i, text, ha="center", va="center", fontsize=9, color="black")
ax.set_title("Final-site agreement with reference")
fig.tight_layout()
site_path = os.path.join(PLOTS_DIR, "site_agreement_heatmap.png")
fig.savefig(site_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display_inline(site_path)
print(f"Saved: {os.path.abspath(site_path)}")


## The AdsorbML bias: naive top-site vs batch-minimum

The pedagogical centrepiece: for each pair, compare the *top-site-only* binding energy (the result a single-starting-point protocol would have returned) against the *batch-minimum* from the full configuration search. The gap is the per-pair bias that AdsorbML quantified at the 100–400 meV level.


In [ ]:
bias_rows = []
for (host, adsorbate), df in PAIR_RESULTS.items():
    top_sites = df[df["start_site"].isin({"top", "al-top"})]
    if len(top_sites) == 0:
        continue  # pair didn't include a top-like site
    naive = top_sites["E_bind (eV)"].min()  # best of top-only starts
    batch = df["E_bind (eV)"].min()
    bias_rows.append({
        "pair": f"{adsorbate}/{host}",
        "naive top-only (eV)": round(naive, 3),
        "batch minimum (eV)":   round(batch, 3),
        "bias (meV)":           int(round((naive - batch) * 1000)),
    })
bias_df = pd.DataFrame(bias_rows)

fig, ax = plt.subplots(figsize=(8, max(2 + 0.4 * len(bias_df), 3)))
if len(bias_df):
    y = range(len(bias_df))
    ax.barh([yi - 0.18 for yi in y], bias_df["naive top-only (eV)"], 0.35,
            color="#d62728", label="naive top-site only")
    ax.barh([yi + 0.18 for yi in y], bias_df["batch minimum (eV)"], 0.35,
            color="#1f77b4", label="AdsorbML batch minimum")
    ax.set_yticks(list(y))
    ax.set_yticklabels(bias_df["pair"])
    ax.axvline(0, color="k", lw=0.5)
    ax.set_xlabel("E_bind (eV)  —  more negative = stronger binding")
    ax.set_title("AdsorbML bias: single-start vs batch search")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, axis="x", ls="--", alpha=0.4)
    # annotate bias
    for i, r in bias_df.reset_index().iterrows():
        ax.text(max(r["naive top-only (eV)"], r["batch minimum (eV)"]) + 0.02,
                i, f"Δ = {r['bias (meV)']:+d} meV", va="center", fontsize=9)

fig.tight_layout()
bias_path = os.path.join(PLOTS_DIR, "adsorbml_bias.png")
fig.savefig(bias_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display_inline(bias_path)
print(f"Saved: {os.path.abspath(bias_path)}")
bias_df


## Discovery plot: 9 pairs coloured by tier

x-axis: MACE batch-minimum E_bind. Triangles: MACE. Diamonds: published DFT reference. Error bar: +/- MAD (0.28 eV for MACE-MPA-0+D3 on the OC157 molecule-surface relative-energy task). Colour coding by pivot-doc tier (validation / discovery / discrepancy).

In [ ]:
tier_color = {"validation": "#2ca02c", "discovery": "#1f77b4",
               "discrepancy": "#d62728", "?": "#888888"}
disc_df = summary_df.sort_values("E_MACE_eV").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.5, max(3, 0.5 * len(disc_df) + 2)))
y = list(range(len(disc_df)))
for i, r in disc_df.iterrows():
    c = tier_color.get(r["tier"], "#888")
    ax.plot(r["E_MACE_eV"], i, "^", color=c, markersize=13, zorder=3,
            label=f"MACE ({r['tier']})" if i == 0 else None)
    is_parity_ref = r["reference_scope"] in {"strict", "near-strict"}
    if pd.notna(r["E_ref_eV"]):
        ref_color = c if is_parity_ref else "#777777"
        ax.plot(r["E_ref_eV"], i, "D", color=ref_color, markersize=9, zorder=3,
                markerfacecolor="white")
        if is_parity_ref:
            ax.plot([r["E_ref_eV"] - MACE_MPA0_OC157_MAD_EV,
                     r["E_ref_eV"] + MACE_MPA0_OC157_MAD_EV],
                    [i, i], "-", color=c, lw=2, alpha=0.35, zorder=1)

ax.set_yticks(y)
ax.set_yticklabels([f"{r['pair']} [{r['status']}]" for _, r in disc_df.iterrows()])
ax.set_xlabel("E_bind (eV)   —   more negative = stronger binding")
ax.set_title("AdsorbML discovery plot  ·  △ MACE-MPA-0  ◇ parity/context reference  ·  MAD bars only for parity rows")
ax.axvline(0, color="k", lw=0.5)
ax.grid(True, axis="x", ls="--", alpha=0.4)
import matplotlib.patches as mpatches
legend_handles = [mpatches.Patch(color=c, label=t) for t, c in tier_color.items() if t != "?"]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)
fig.tight_layout()
disc_path = os.path.join(PLOTS_DIR, "discovery_plot.png")
fig.savefig(disc_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display_inline(disc_path)
print(f"Saved: {os.path.abspath(disc_path)}")


In [ ]:
import json

RESULT_TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(RESULT_TABLE_DIR, exist_ok=True)

pair_result_paths = []
for (host, adsorbate), df in PAIR_RESULTS.items():
    path = os.path.join(RESULT_TABLE_DIR, f"pair_results_{_safe(adsorbate)}_{_safe(host)}.csv")
    df.to_csv(path, index=False)
    pair_result_paths.append(path)

summary_path = os.path.join(RESULT_TABLE_DIR, "summary_validation.csv")
summary_df.to_csv(summary_path, index=False)

bias_path_csv = os.path.join(RESULT_TABLE_DIR, "adsorbml_bias.csv")
bias_df.to_csv(bias_path_csv, index=False)

metadata_path = os.path.join(RESULT_TABLE_DIR, "run_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "backend": BACKEND,
            "small_panel_mode": SMALL_PANEL_MODE,
            "toolkit_checkpoint": TOOLKIT_CHECKPOINT,
            "toolkit_device": str(getattr(RELAXATION_BACKEND, "device", TOOLKIT_DEVICE)),
            "toolkit_dtype": TOOLKIT_DTYPE,
            "toolkit_compile_model": TOOLKIT_COMPILE_MODEL,
            "toolkit_n_steps": TOOLKIT_N_STEPS,
            "toolkit_fmax": TOOLKIT_FMAX,
            "toolkit_d3bj_enabled": TOOLKIT_D3BJ is not None,
            "toolkit_require_d3bj": TOOLKIT_REQUIRE_D3BJ,
            "require_visrtx_render": REQUIRE_VISRTX_RENDER,
            "visrtx_render_failures": visrtx_render_failures,
            "e_gas_ads_ev": E_ADS_GAS,
            "e_host_ev": E_HOST,
            "pair_result_paths": pair_result_paths,
            "summary_path": summary_path,
            "bias_path": bias_path_csv,
        },
        f,
        indent=2,
    )

print("Saved result tables:")
for path in [*pair_result_paths, summary_path, bias_path_csv, metadata_path]:
    print(f"  {os.path.abspath(path)}")


---

## Discovery narrative

**Validation candidates.** These are pairs where the batch minimum agrees with a well-supported reference site and, once a strict or near-strict reference row is available, can be compared quantitatively against the published energy. The current notebook keeps these rows contextual until exact OC20, OC22, AdsorbML, or primary-literature records are recorded in `references/manifest.yml`.

**Configuration-search cases.** These are pairs where a narrow single-start protocol would report a higher-energy local minimum than the batched search. CO/Pd(111) is the clearest intended example because hollow-site binding is well established as surface-science context, but the exact energy comparison remains contextual until the matching slab and coverage are pinned.

**Review cases.** These are pairs where the final site, adsorption energy, or reference assignment is ambiguous. The correct scientific response is not to force agreement, but to identify which assumption needs review: the MLIP relaxation, the geometry classifier, the reference row, or the chemical model itself.

## Key methodological takeaway

Configuration search changes an adsorption calculation from a test of one starting geometry into a controlled comparison across plausible local minima. Batch GPU relaxation makes that comparison practical, but the resulting numbers still require reference-aware interpretation.

## Scope limits

- **Activation barriers** are not computed. Thermodynamic binding is not a kinetic pathway; NEB, dimer, or string methods are outside this tutorial.
- **Coverage-dependent lateral interactions** are absent. Every configuration here is a single adsorbate at low coverage.
- **Temperature and entropy corrections** are absent. The reported quantities are electronic adsorption energies, not finite-temperature free energies.
- **Open-shell, magnetic 3d, reducible-oxide defect, and f-electron chemistry** are excluded from the active panel.
- **D3(BJ) is part of the deployed model/service convention.** Any strict reference comparison must verify the reference dispersion convention.
- **OC20/OC22/literature values labelled `context` are not strict parity rows.** They must stay out of apples-to-apples error statistics until exact slab, coverage, functional, dispersion, frozen-layer, and sign-convention metadata are recorded in `references/manifest.yml`.

## References

1. Batatia, I. *et al.* "A foundation model for atomistic materials chemistry." arXiv:[2401.00096](https://arxiv.org/abs/2401.00096). Model-level MACE uncertainty source; exact table/version must be checked before strict numerical claims.
2. Lan, J. *et al.* "AdsorbML: a leap in efficiency for adsorption energy calculations using generalizable machine learning potentials." *npj Computational Materials* **9**, 172 (2023). DOI: [10.1038/s41524-023-01121-5](https://doi.org/10.1038/s41524-023-01121-5).
3. Chanussot, L. *et al.* "Open Catalyst 2020 (OC20) Dataset and Community Challenges." *ACS Catalysis* **11**, 6059 (2021). DOI: [10.1021/acscatal.0c04525](https://doi.org/10.1021/acscatal.0c04525).
4. Tran, R. *et al.* "The Open Catalyst 2022 (OC22) Dataset and Challenges for Oxide Electrocatalysts." *ACS Catalysis* **13**, 3066 (2023). DOI: [10.1021/acscatal.2c05426](https://doi.org/10.1021/acscatal.2c05426).
5. Hammer, B., Morikawa, Y. and Norskov, J. K. "CO chemisorption at metal surfaces and overlayers." *Physical Review Letters* **76**, 2141 (1996). DOI: [10.1103/PhysRevLett.76.2141](https://doi.org/10.1103/PhysRevLett.76.2141).
6. Grimme, S. *et al.* "Effect of the damping function in dispersion corrected density functional theory." *Journal of Computational Chemistry* **32**, 1456 (2011). DOI: [10.1002/jcc.21759](https://doi.org/10.1002/jcc.21759).
7. Stukowski, A. "Visualization and analysis of atomistic simulation data with OVITO." *Modelling and Simulation in Materials Science and Engineering* **18**, 015012 (2010). DOI: [10.1088/0965-0393/18/1/015012](https://doi.org/10.1088/0965-0393/18/1/015012).

Additional pair-level citations remain in `helpers/references.py` as context rows and in `references/manual_checks.md` as verification tasks.

The previous AWH water-sorbent materials are archived under [`_archive/awh-pivot-sources/`](_archive/awh-pivot-sources/), and the previous OER catalyst-screening tutorial is archived under [`_archive/oer-catalyst-screening/`](_archive/oer-catalyst-screening/).

---

*End of notebook.*
